# AI track: Bedrock application plumbing

You build the request side of a generative-AI feature: prompts, streaming, tool use, token budgeting and error handling.

**Note:** no language model runs in the emulator. Replies are placeholders, but requests are validated like the real API and tokens are counted, so you can develop and test the integration without paying for inference. Answer quality can only be judged on real Bedrock.

In [1]:
import sys, json, time, datetime as dt
sys.path.insert(0, "../..")          # repository root: emu.py, lab_common.py
import pandas as pd
import matplotlib.pyplot as plt
import emu
from lab_common import *

df = ensure_lake()                   # buckets, loan book in S3, Glue table (safe to re-run)
print("Connected to", emu.settings()["endpoint_url"], "-", len(df), "loans in the lake")

Connected to https://127.0.0.1:9297 - 2000 loans in the lake


In [2]:
bedrock = emu.client("bedrock-runtime")
MODEL = "anthropic.claude-3-haiku-20240307-v1:0"
by_segment = run_sql("SELECT segment, AVG(CAST(defaulted AS DOUBLE)) AS default_rate FROM loans GROUP BY segment")
facts = by_segment.round(3).to_dict("records")
facts

[{'segment': 'Agri', 'default_rate': 0.212},
 {'segment': 'SME', 'default_rate': 0.209},
 {'segment': 'Salaried', 'default_rate': 0.24},
 {'segment': 'Self-employed', 'default_rate': 0.272}]

## 1. A grounded question with Converse

Portfolio facts from Athena go into the prompt. Look at the shape of the response: message, stop reason and usage.

In [3]:
resp = attempt("Converse", lambda: bedrock.converse(
    modelId=MODEL,
    system=[{"text": "You are a credit-risk analyst. Answer in two sentences."}],
    messages=[{"role": "user", "content": [{"text": f"Segment default rates: {facts}. Which segment needs tighter underwriting?"}]}],
    inferenceConfig={"maxTokens": 200, "temperature": 0.2}))
print("Reply      :", resp["output"]["message"]["content"][0]["text"])
print("Stop reason:", resp["stopReason"])
print("Usage      :", resp["usage"])

OK      Converse
Reply      : No language model runs in this emulator. anthropic.claude-3-haiku-20240307-v1:0 generated no text for this request. The request carried 1 message(s) and 89 input token(s).
Stop reason: end_turn
Usage      : {'inputTokens': 89, 'outputTokens': 48, 'totalTokens': 137}


## 2. Streaming

A chat interface reads the reply as events. The event sequence is the same one real Bedrock sends.

In [4]:
stream = bedrock.converse_stream(modelId=MODEL, messages=[{"role": "user", "content": [{"text": "Summarise portfolio risk."}]}])["stream"]
kinds, text = [], ""
for event in stream:
    kind = next(iter(event)); kinds.append(kind)
    if kind == "contentBlockDelta":
        text += event[kind]["delta"].get("text", "")
print(pd.Series(kinds).value_counts().to_dict())
print(text)

{'contentBlockDelta': 12, 'messageStart': 1, 'contentBlockStop': 1, 'messageStop': 1, 'metadata': 1}
No language model runs in this emulator. anthropic.claude-3-haiku-20240307-v1:0 generated no text for this request. The request carried 1 message(s) and 6 input token(s).


## 3. Tool use

With `toolChoice` set to `any`, the model must call a tool. The emulator returns a `toolUse` block shaped by your schema, which lets you test your tool-dispatch code.

In [5]:
tools = {"tools": [{"toolSpec": {"name": "flag_segment", "description": "Flag a customer segment for tighter underwriting",
          "inputSchema": {"json": {"type": "object", "properties": {"segment": {"type": "string"}, "reason": {"type": "string"}}, "required": ["segment"]}}}}],
         "toolChoice": {"any": {}}}
resp = attempt("Converse with tools", lambda: bedrock.converse(modelId=MODEL, toolConfig=tools,
    messages=[{"role": "user", "content": [{"text": f"Given {facts}, flag the riskiest segment."}]}]))
call = resp["output"]["message"]["content"][0]["toolUse"]
print("Stop reason:", resp["stopReason"])
print("Tool call  :", call["name"], call["input"])

def flag_segment(segment, reason=""):          # your own function behind the tool
    return {"flagged": segment, "ticket": "RISK-1001"}
print("Dispatched :", {"flag_segment": flag_segment}[call["name"]](**call["input"]))

OK      Converse with tools
Stop reason: tool_use
Tool call  : flag_segment {'segment': 'segment'}
Dispatched : {'flagged': 'segment', 'ticket': 'RISK-1001'}


## 4. Token budgeting

Count tokens before sending, to estimate cost per request. The price below is an illustrative figure, not a quoted rate.

In [6]:
PRICE_PER_1K_INPUT = 0.00025   # illustrative only; check current Bedrock pricing
rows = []
for n in (5, 50, 500):
    prompt = "Assess these loans: " + df.head(n).to_json(orient="records")
    tokens = bedrock.count_tokens(modelId=MODEL, input={"converse": {"messages": [{"role": "user", "content": [{"text": prompt}]}]}})["inputTokens"]
    rows.append({"loans_in_prompt": n, "input_tokens": tokens, "est_cost_usd": round(tokens / 1000 * PRICE_PER_1K_INPUT, 5)})
pd.DataFrame(rows)

,loans_in_prompt,input_tokens,est_cost_usd
0,5,318,0.00008
1,50,3129,0.00078
2,500,31242,0.00781


## 5. Error handling

A wrong model identifier is refused with the same error class as real Bedrock, so your `except` branches can be tested.

In [7]:
attempt("Converse with a bad model ID", lambda: bedrock.converse(modelId="not-a-model", messages=[{"role": "user", "content": [{"text": "hi"}]}]))

REFUSED Converse with a bad model ID: ValidationException - The provided model identifier is invalid.


## Exercises

1. Send the tool result back as a `toolResult` message and complete the two-turn loop.
2. Add a `stopSequences` entry and observe the stop reason.
3. Wrap `converse` in a function that refuses any prompt above a token budget you choose.